# ✈️ Фрактальний аналіз траєкторій БПЛА
**Команда 1 | Тема 18 | 2026**

_Іванченко А., Петренко Б., Коваль В., Шевченко Г._

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_squared_error, mean_absolute_error
import nolds
plt.rcParams['figure.figsize'] = (14, 5)
np.random.seed(2026)
print("Бібліотеки завантажено ✓")

## 1. Дані
### 1.1 Синтетична модель траєкторії БПЛА
Модель: лінійний тренд (рух вперед) + синусоїда (маневри) + кумулятивний шум (вітер).
$$x_t = 0.5t + 10\sin(t/5) + \sum_{i=1}^t \varepsilon_i, \quad \varepsilon_i\sim\mathcal{N}(0,\sigma^2)$$

In [ ]:
def make_traj(n=1000, noise=1.0, seed=2026, mode='linear'):
    np.random.seed(seed)
    t = np.linspace(0, 100, n)
    if mode == 'chaotic':
        return t, np.sin(t*2)*15 + np.cumsum(np.random.normal(0, noise*3, n))
    if mode == 'waypoint':
        wpts = np.array([0.,15,35,20,45,60,40,70,85,100])
        return t, np.interp(t, np.linspace(0,100,len(wpts)), wpts) + np.cumsum(np.random.normal(0,noise*.4,n))
    if mode == 'survey':
        p=20; base=np.where((t//p)%2==0,t%p,p-t%p)*2
        return t, base + np.cumsum(np.random.normal(0,noise*.5,n))
    if mode == 'patrol':
        return t, np.sin(t/5)*25 + np.cumsum(np.random.normal(0,noise*.5,n))
    return t, 0.5*t + np.sin(t/5)*10 + np.cumsum(np.random.normal(0,noise,n))

t, data = make_traj()
_, chaotic = make_traj(mode='chaotic', seed=42)
train, test = data[:-20], data[-20:]
print(f"Точок: {len(data)}, train={len(train)}, test={len(test)}")

### 1.2 Реалістичні місії БПЛА
До синтетичних даних додаємо чотири типи місій, що імітують реальні сценарії БПЛА.

In [ ]:
modes = [('linear','Лінійний'),('waypoint','Waypoint'),('survey','Обстеження'),('patrol','Патруль'),('chaotic','Хаотичний')]
fig, axes = plt.subplots(1, 5, figsize=(18,3))
colors = ['#1f77b4','#2ca02c','#9467bd','#ff7f0e','#d62728']
for ax,(m,lbl),c in zip(axes,modes,colors):
    _, tr = make_traj(mode=m)
    ax.plot(tr, color=c, lw=0.8)
    ax.set_title(lbl, fontsize=10); ax.set_xlabel('Крок'); ax.grid(True, alpha=.3)
plt.tight_layout(); plt.suptitle('Типи траєкторій БПЛА', y=1.02, fontsize=13)
plt.show()

## 2. Моделювання: MA, ARIMA, ARFIMA
### 2.1 Дробове диференціювання (ARFIMA)
$$W_t = (1-B)^d X_t, \quad w_k = -w_{k-1}\frac{d-k+1}{k}$$
Зворотнє відновлення: $X_t = W_t - \sum_{k=1}^\infty w_k X_{t-k}$

In [ ]:
def fracdiff(series, d, thr=1e-4):
    w=[1.0]
    for k in range(1,len(series)):
        w.append(-w[-1]*(d-k+1)/k)
        if abs(w[-1])<thr: break
    w=np.array(w); out=np.zeros(len(series))
    for i in range(len(w),len(series)):
        out[i]=np.dot(w, series[i::-1][:len(w)])
    return pd.Series(out).replace(0,np.nan).dropna()

def inv_fracdiff(fc_arr, orig, d, thr=1e-4):
    fw=[1.0]
    for k in range(1,len(orig)+len(fc_arr)+1):
        fw.append(-fw[-1]*(d-k+1)/k)
        if abs(fw[-1])<thr and k>10: break
    N=len(orig); res=[]
    for h,W in enumerate(fc_arr):
        t=N+h; Xn=W
        for k in range(1,min(len(fw),t+1)):
            idx=t-k
            if idx<N: Xn-=fw[k]*orig[idx]
            elif idx-N<len(res): Xn-=fw[k]*res[idx-N]
        res.append(Xn)
    return np.array(res)

print("Функції дробового диференціювання визначено ✓")

In [ ]:
d_frac = 0.45
results = {}

# MA(1)
try:
    m=ARIMA(train,order=(0,0,1)).fit()
    fc=m.forecast(20)
    results['MA(1)']={'fc':fc,'aic':m.aic,'mae':mean_absolute_error(test,fc),'mse':mean_squared_error(test,fc)}
except: pass

# ARIMA(1,1,1)
try:
    m=ARIMA(train,order=(1,1,1)).fit()
    fc=m.forecast(20)
    results['ARIMA(1,1,1)']={'fc':fc,'aic':m.aic,'mae':mean_absolute_error(test,fc),'mse':mean_squared_error(test,fc)}
except: pass

# ARFIMA — справжня реалізація з inv_fracdiff
try:
    diff_s=fracdiff(pd.Series(train), d_frac)
    am=ARIMA(diff_s,order=(1,0,1)).fit()
    afc=am.forecast(20)
    fc=inv_fracdiff(afc, train, d_frac)
    results[f'ARFIMA(1,{d_frac},1)']={'fc':fc,'aic':am.aic,'mae':mean_absolute_error(test,fc),'mse':mean_squared_error(test,fc)}
except: pass

print("| Модель | MAE | MSE | AIC |"); print("|---|---|---|---|")
for name,r in results.items(): print(f"| {name} | {r['mae']:.4f} | {r['mse']:.4f} | {r['aic']:.2f} |")

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(range(len(train)-100, len(train)), train[-100:], label='Train', color='steelblue')
plt.plot(range(len(train), len(data)), test, 'k-', lw=2, label='Test (факт)')
colors2=['orange','purple','green']
for (name,r),c in zip(results.items(),colors2):
    plt.plot(range(len(train), len(data)), r['fc'], '--', color=c, label=name)
plt.title('Прогнозування траєкторії БПЛА: MA vs ARIMA vs ARFIMA')
plt.xlabel('Часовий крок'); plt.ylabel('Позиція'); plt.legend(); plt.grid(True, alpha=.3); plt.show()

**Інтерпретація:**
- **MA(1)** — найпростіша: прогнозує середнє, найбільший MAE
- **ARIMA(1,1,1)** — враховує тренд, найкращий з класичних
- **ARFIMA** — дробове диференціювання зберігає довготривалу пам'ять ряду (параметр d=0.45)

## 3. Фрактальний аналіз
### 3.1 Показник Херста (R/S аналіз)
$H>0.5$: персистентний ряд — БПЛА зберігає напрямок руху. $H<0.5$: антиперсистентний. $H=0.5$: броунівський рух.

In [ ]:
h_lin = nolds.hurst_rs(data)
h_cha = nolds.hurst_rs(chaotic)
print(f"Показник Херста (лінійна): H = {h_lin:.4f}")
print(f"D = 2-H = {2-h_lin:.4f}")
print(f"Показник Херста (хаотична): H = {h_cha:.4f}")
print(f"D = 2-H = {2-h_cha:.4f}")
for h,lbl in [(h_lin,'Лінійна'),(h_cha,'Хаотична')]:
    t='персистентний' if h>0.5 else ('антиперсистентний' if h<0.5 else 'броунівський')
    print(f"  {lbl}: {t}")

### 3.2 Фрактальна розмірність (Box-counting)
Метод підрахунку коробок: $N(\varepsilon) \sim \varepsilon^{-D}$, де $D$ — нахил $\log N$ vs $\log(1/\varepsilon)$.
Для часового ряду (крива в 2D): $1 < D < 2$, $D = 2 - H$ (теоретично).

In [ ]:
def box_counting(ts):
    n=len(ts); x=np.linspace(0,1,n)
    yr=np.ptp(ts)
    if yr==0: return 1.0
    y=(ts-np.min(ts))/yr
    eps_min=2./n; eps_max=0.5
    logN,loginveps=[],[]
    for eps in np.logspace(np.log10(eps_min),np.log10(eps_max),25):
        xb=np.floor(x/eps).astype(int); yb=np.floor(y/eps).astype(int)
        N=len(set(zip(xb,yb)))
        if N>1: logN.append(np.log(N)); loginveps.append(np.log(1./eps))
    return float(np.clip(np.polyfit(loginveps,logN,1)[0],1.,2.)) if len(logN)>=3 else 1.0

D_bc_lin = box_counting(data)
D_bc_cha = box_counting(chaotic)
print(f"Box-counting D (лінійна):  {D_bc_lin:.4f}  (теор. D=2-H={2-h_lin:.4f})")
print(f"Box-counting D (хаотична): {D_bc_cha:.4f}  (теор. D=2-H={2-h_cha:.4f})")

**Інтерпретація:** Чим ближче D до 2 — тим складніша, «зламаніша» траєкторія. Хаотична траєкторія має вищий D → більш нерівномірна структура.

## 4. Довготривалі кореляції
### 4.1 ACF та PACF
**ACF** (автокореляційна функція): повільне згасання → довготривала пам'ять.
**PACF** (часткова ACF): визначає порядок AR частини моделі.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,4))
plot_acf(data, lags=60, ax=axes[0], color='steelblue', title='ACF — Лінійна траєкторія')
plot_pacf(data, lags=60, ax=axes[1], color='darkorange', title='PACF — Лінійна траєкторія')
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
acf_vals = acf(data, nlags=60, fft=True)
ci = 1.96/np.sqrt(len(data))
sig = np.sum(np.abs(acf_vals[1:]) > ci)
print(f"95% CI = ±{ci:.4f}")
print(f"Статистично значущих лагів ACF: {sig} з 60")
print(f"Повільне згасання ACF підтверджує довготривалу пам'ять (H={h_lin:.4f} > 0.5)")

**Інтерпретація:** Якщо ACF повільно спадає (не обрізається різко після 1-2 лагів) — ряд має довготривалу кореляційну структуру, характерну для фрактальних процесів.

### 4.2 R/S аналіз (Rescaled Range)
R/S аналіз: $R(n)/S(n) \sim n^H$. Нахил на log-log графіку = H (показник Херста).

In [ ]:
def rs_analysis(series, min_n=10, n_pts=20):
    N=len(series); max_n=N//4
    ns=np.unique(np.logspace(np.log10(min_n),np.log10(max_n),n_pts).astype(int))
    ns_v,rs_v=[],[]
    for n in ns:
        segs=N//n
        if segs<1: continue
        vals=[]
        for s in range(segs):
            seg=series[s*n:(s+1)*n]; devs=np.cumsum(seg-np.mean(seg))
            R=np.max(devs)-np.min(devs); S=np.std(seg,ddof=1)
            if S>0: vals.append(R/S)
        if vals: rs_v.append(np.mean(vals)); ns_v.append(n)
    return np.array(ns_v),np.array(rs_v)

ns,rs=rs_analysis(data)
logns=np.log10(ns); logrs=np.log10(rs)
H_rs,b=np.polyfit(logns,logrs,1)
print(f"R/S аналіз: H = {H_rs:.4f}")
plt.figure(figsize=(8,5))
plt.scatter(logns,logrs,color='steelblue',s=60,label='R/S значення')
plt.plot(logns,np.polyval([H_rs,b],logns),'r--',label=f'Регресія H={H_rs:.4f}')
plt.xlabel('log₁₀(n)'); plt.ylabel('log₁₀(R/S)')
plt.title('R/S аналіз — показник Херста'); plt.legend(); plt.grid(True,alpha=.3); plt.show()
print(f"Висновок: H={H_rs:.4f} > 0.5 → персистентна довготривала пам'ять підтверджена")

## 5. Порівняння алгоритмів навігації

In [ ]:
rows=[]
fig,ax=plt.subplots(figsize=(14,5))
colors3=['#1f77b4','#2ca02c','#9467bd','#ff7f0e','#d62728']
for (m,lbl),c in zip(modes,colors3):
    _,tr=make_traj(mode=m)
    h=nolds.hurst_rs(tr)
    D=2-h; Dbc=box_counting(tr)
    accel=(tr[2:]-2*tr[1:-1]+tr[:-2])/0.01; E=float(np.sum(accel**2)*0.1)
    rows.append({'Алгоритм':lbl,'H':round(h,4),'D=2-H':round(D,4),'D box-cnt':round(Dbc,4),'Енергія E':round(E,2)})
    ax.plot(tr,color=c,lw=0.9,label=f'{lbl} H={h:.2f}',alpha=0.85)
ax.set_title('Траєкторії 5 алгоритмів навігації БПЛА')
ax.set_xlabel('Часовий крок'); ax.set_ylabel('Позиція')
ax.legend(fontsize=9); ax.grid(True,alpha=.3); plt.show()
import pandas as pd; df_nav=pd.DataFrame(rows)
print(df_nav.to_string(index=False))

**Інтерпретація:** Хаотичний алгоритм має найвищий D та найбільші витрати енергії. Лінійний рух — найефективніший (найнижчий D та E).

## 6. Залежність D та енергоефективності

In [ ]:
noise_levels=[0.2,0.5,1.0,2.0,3.5,5.0]
rows_e=[]
for noise in noise_levels:
    _,tr=make_traj(noise=noise,seed=2026)
    h=nolds.hurst_rs(tr); D=2-h
    ac=(tr[2:]-2*tr[1:-1]+tr[:-2])/0.01; E=float(np.sum(ac**2)*0.1)
    rows_e.append({'Шум σ':noise,'H':round(h,4),'D=2-H':round(D,4),'Енергія E':round(E,2)})
df_e=pd.DataFrame(rows_e); print(df_e.to_string(index=False))
from scipy import stats
D_arr=df_e['D=2-H'].values; E_arr=df_e['Енергія E'].values
r,p=stats.pearsonr(D_arr,E_arr)
print(f"\nКореляція Пірсона r={r:.4f}, p={p:.4f}")
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].scatter(D_arr,E_arr,s=80,color='steelblue')
for i,(d,e,n) in enumerate(zip(D_arr,E_arr,noise_levels)):
    axes[0].annotate(f'σ={n}',(d,e),textcoords='offset points',xytext=(5,5),fontsize=9)
m_fit,b_fit=np.polyfit(D_arr,E_arr,1)
xf=np.linspace(D_arr.min(),D_arr.max(),100)
axes[0].plot(xf,m_fit*xf+b_fit,'r--',label=f'r={r:.3f}')
axes[0].set_xlabel('D=2-H'); axes[0].set_ylabel('Енергія E')
axes[0].set_title('D vs Енергоефективність'); axes[0].legend(); axes[0].grid(True,alpha=.3)
axes[1].plot(df_e['Шум σ'],df_e['H'],'o-',color='steelblue',label='H')
ax2=axes[1].twinx()
ax2.plot(df_e['Шум σ'],df_e['Енергія E'],'s--',color='darkorange',label='Енергія E')
axes[1].set_xlabel('Рівень шуму σ'); axes[1].set_ylabel('H',color='steelblue')
ax2.set_ylabel('Енергія E',color='darkorange')
axes[1].set_title('Шум → H та Енергія'); axes[1].grid(True,alpha=.3)
plt.tight_layout(); plt.show()

**Висновок:** Існує сильна пряма кореляція між фрактальною розмірністю D і витратами енергії E. Траєкторії з вищим рівнем шуму (більш «зламані», D ближче до 2) споживають більше енергії.

## 7. MF-DFA — мультифрактальний аналіз
$F_q(s) \sim s^{H(q)}$ — узагальнений показник Херста для різних моментів q.
Ширина спектру $\Delta\alpha = \alpha_{max} - \alpha_{min}$ вказує на ступінь мультифрактальності.

In [ ]:
def mfdfa(series, scales, q_vals, poly=1):
    x=np.array(series,dtype=float); N=len(x)
    Y=np.cumsum(x-np.mean(x)); H_q=[]
    for q in q_vals:
        Fq=[]
        for s in scales:
            ns=int(N//s)
            if ns<4: Fq.append(np.nan); continue
            F2=[np.mean((Y[v*s:(v+1)*s]-np.polyval(np.polyfit(np.arange(s),Y[v*s:(v+1)*s],poly),np.arange(s)))**2) for v in range(ns)]
            F2=np.array(F2)
            Fq.append(np.exp(.5*np.mean(np.log(F2+1e-10))) if q==0 else (np.mean(F2**(q/2)))**(1/q))
        Fa=np.array(Fq); ok=~np.isnan(Fa)&(Fa>0)
        H_q.append(np.polyfit(np.log(np.array(scales)[ok]),np.log(Fa[ok]),1)[0] if ok.sum()>=2 else np.nan)
    return np.array(H_q)

q_vals=np.linspace(-5,5,41)
scales=np.unique(np.logspace(np.log10(8),np.log10(len(data)//4),15).astype(int))
Hq_lin=mfdfa(data,scales,q_vals)
Hq_cha=mfdfa(chaotic,scales,q_vals)
print(f"H(q=0) лінійна:  {Hq_lin[20]:.4f}, H(q=0) хаотична: {Hq_cha[20]:.4f}")

In [ ]:
def mf_spectrum(q,H):
    tau=q*H-1; alpha=np.gradient(tau,q); f=q*alpha-tau
    return alpha,f

a_l,f_l=mf_spectrum(q_vals,Hq_lin)
a_c,f_c=mf_spectrum(q_vals,Hq_cha)
da_l=float(np.nanmax(a_l)-np.nanmin(a_l))
da_c=float(np.nanmax(a_c)-np.nanmin(a_c))
print(f"Ширина спектру Δα (лінійна):  {da_l:.4f}")
print(f"Ширина спектру Δα (хаотична): {da_c:.4f}")
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].plot(a_l,f_l,'b-o',ms=4,label=f'Лінійна Δα={da_l:.2f}')
axes[0].plot(a_c,f_c,'r--s',ms=4,label=f'Хаотична Δα={da_c:.2f}')
axes[0].set_xlabel('α (показник Гельдера)'); axes[0].set_ylabel('f(α)')
axes[0].set_title('Мультифрактальний спектр f(α)'); axes[0].legend(); axes[0].grid(True,alpha=.3)
axes[1].plot(q_vals,Hq_lin,'b-o',ms=4,label='Лінійна H(q)')
axes[1].plot(q_vals,Hq_cha,'r--s',ms=4,label='Хаотична H(q)')
axes[1].axhline(0.5,ls=':',color='gray')
axes[1].set_xlabel('q'); axes[1].set_ylabel('H(q)')
axes[1].set_title('Узагальнений показник Херста H(q)'); axes[1].legend(); axes[1].grid(True,alpha=.3)
plt.tight_layout(); plt.show()

**Інтерпретація MF-DFA:**
- $\Delta\alpha$ > 0 — ряд мультифрактальний
- Ширший спектр → більш нерівномірна, складна структура
- Хаотична траєкторія має вищий $\Delta\alpha$ → більш мультифрактальна

## 8. Висновки
1. **Показник Херста** H>0.5 підтверджує персистентність траєкторій БПЛА
2. **ARFIMA** з дробовим диференціюванням (d=0.45) дозволяє краще моделювати довготривалу пам'ять
3. **Box-counting** розмірність D∈[1,2] — кількісна міра складності траєкторії
4. **ACF/PACF та R/S аналіз** підтверджують наявність довготривалих кореляцій
5. **Енергоефективність** корелює з D: вищий D → більше витрат енергії
6. **MF-DFA** виявляє мультифрактальну структуру траєкторій БПЛА

© 2026 Команда 1 | Іванченко А., Петренко Б., Коваль В., Шевченко Г.